In [1]:
!huggingface-cli download Qwen/Qwen3-0.6B --local-dir ./qwen3-0.6b


⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 10 files:   0% 0/10 [00:00<?, ?it/s]Downloading 'README.md' to 'qwen3-0.6b/.cache/huggingface/download/Xn7B-BWUGOee2Y6hCZtEhtFu4BE=.a50b19e76f5274f9ec99f5a5d99873dca5bff25e.incomplete'

README.md: 14.0kB [00:00, 30.3MB/s]
Download complete. Moving file to qwen3-0.6b/README.md

generation_config.json: 100% 239/239 [00:00<00:00, 1.87MB/s]
Download complete. Moving file to qwen3-0.6b/generation_config.json

merges.txt: 0.00B [00:00, ?B/s]Downloading 'config.json' to 'qwen3-0.6b/.cache/huggingface/download/8_PA_wEVGiVa2goH2H4KQOQpvVY=.f5c3703b78ae2a478ae15b247e9f855e0ce2107b.incomplete'


.gitattributes: 1.57kB [00:00, 9.70MB/s]
Download complete. Moving file to qwen3-0.6b/.gitattributes
merges.txt: 1.67MB [00:00, 73.1MB/s]
Download complete. Moving file to qwen3-0.6b/merges.txt
Fetching 10 files:  10% 1/10 [00:00<00:03,  2.70it/s]Downloading 'tokenizer_config.json' to 'qwen3-0.6b/.cache/huggingface/

In [2]:
# transformers 라이브러리에서 토크나이저와 언어 모델 로드를 위한 클래스들을 가져옵니다.
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. 토크나이저 로드: 텍스트를 모델이 이해할 수 있는 숫자 형태로 변환하는 도구입니다.
# "/content/qwen3-0.6b" 경로에 저장된 설정값을 바탕으로 토크나이저를 불러옵니다.
tokenizer = AutoTokenizer.from_pretrained("/content/qwen3-0.6b")

# 2. 모델 로드: 실제 추론(텍스트 생성)을 수행하는 인공지능 모델 본체를 불러옵니다.
# AutoModelForCausalLM은 GPT처럼 이전 단어를 바탕으로 다음 단어를 예측하는 모델 구조에 쓰입니다.
model = AutoModelForCausalLM.from_pretrained("/content/qwen3-0.6b")

In [3]:
# 모델의 내부 네트워크 구조와 레이어 구성을 터미널(콘솔)에 출력합니다.
# 어떤 블록(Attention, MLP 등)이 몇 개 쌓여 있는지, 파라미터 구성은 어떤지 확인할 때 사용합니다.
print(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [4]:
# 1. 모델의 모든 가중치와 편향(Parameter) 데이터가 담긴 사전(dict)을 가져옵니다.
# state_dict()는 레이어 이름(key)과 수치값(tensor)을 쌍으로 가지고 있습니다.
params = model.state_dict()

# 2. 모든 파라미터의 이름을 하나씩 훑어봅니다.
for name, param in params.items():
    # print(name)  # (주석 처리됨) 모든 레이어 이름을 보고 싶을 때 주석을 해제합니다.

    # 3. 특정 이름을 포함하는 파라미터만 필터링합니다.
    # 여기서는 '23번 레이어'의 'Self-Attention' 구조 중 'Query 프로젝션'의 '가중치'를 찾습니다.
    if 'model.layers.23.self_attn.q_proj.weight' in name:
        # 찾은 레이어의 이름과 그 안에 저장된 실제 숫자값(Tensor)을 출력합니다.
        print(name, param)

model.layers.23.self_attn.q_proj.weight tensor([[-0.0109, -0.0055,  0.0073,  ...,  0.0957,  0.0302,  0.0408],
        [ 0.0366,  0.0006, -0.0002,  ..., -0.0713,  0.0302,  0.0381],
        [-0.0058, -0.0097, -0.0029,  ...,  0.0052,  0.0136,  0.0006],
        ...,
        [-0.0145, -0.0194, -0.0048,  ..., -0.0166,  0.0258, -0.0162],
        [ 0.0247, -0.0242, -0.0170,  ...,  0.0080, -0.0091,  0.0210],
        [ 0.0219,  0.0162,  0.0016,  ...,  0.0267,  0.0168, -0.0051]])


In [5]:
# !: 주피터 노트북이나 코랩 환경에서 터미널 명령어를 실행하겠다는 기호입니다.
# pip install lm_eval: 언어 모델 평가 프레임워크인 'lm-evaluation-harness' 라이브러리를 설치합니다.
# --user: 시스템 전체 폴더가 아닌, 현재 로그인한 사용자 계정의 라이브러리 폴더에 설치합니다.
# (권한 문제가 발생할 때 유용하며, 환경을 깔끔하게 유지할 수 있습니다.)
!pip install lm_eval --user

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 13.0 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=f295b5a0c451e764f6fe12a7a9f636b4a68528ec38e3a679bbc2a1e03cc69daf
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
  Created wheel for sqlitedict: filename=sqlitedict-2.1.0-py3-none-any.whl size=16862 sha256=4d6b82002c2bb9a9140548e64a8c149b6b7194e7e5b166367d0189d9e619394f
  Stored in directory: /root/.cache/pip/wheels/7a/6f/21

In [6]:
# 1. python3 -m lm_eval: 평가 도구(lm-evaluation-harness)를 실행합니다.
# 2. --model hf: 허깅페이스 방식의 모델을 평가 대상으로 설정합니다.
# 3. --model_args: 모델 세부 설정 (경로, 데이터 타입 자동 설정, 커스텀 코드 허용).
# 4. --tasks mmlu: 다목적 언어 이해 능력(MMLU)을 테스트 항목으로 지정합니다.
# 5. --device cuda:0: 0번 GPU를 사용하여 계산 속도를 높입니다.
# 6. --limit 50: 테스트 속도를 위해 각 과목당 50개의 샘플만 사용합니다.
# 7. --batch_size auto:4: 메모리에 맞춰 한 번에 처리할 양을 자동 조절(기본 4)합니다.

!python3 -m lm_eval \
    --model hf \
    --model_args pretrained=/content/qwen3-0.6b,dtype="auto",trust_remote_code=True \
    --tasks mmlu \
    --device cuda:0 \
    --limit 50 \
    --batch_size auto:4

2026-02-02:04:20:06 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-02-02:04:20:20 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
2026-02-02:04:20:21 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-02:04:20:21 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': '/content/qwen3-0.6b', 'dtype': 'auto', 'trust_remote_code': True}
2026-02-02:04:20:24 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-02-02:04:20:24 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
2026-02-02 04:20:25.399506: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 

In [7]:
# 1. pip install vllm: 고속 추론 엔진인 vLLM 라이브러리를 설치합니다.
# 2. vLLM은 PagedAttention 기술을 사용하여 GPU 메모리 효율을 극대화하고,
#    Hugging Face 기본 로더보다 훨씬 빠른 생성 속도(Throughput)를 제공합니다.
# 3. Qwen3-0.6B처럼 작은 모델도 vLLM을 쓰면 서빙 속도가 매우 빨라집니다.

!pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 556.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 M

In [1]:
# 1. python3 -m vllm.entrypoints.openai.api_server: vllm을 OpenAI API 규격의 서버로 실행합니다.
# 2. --model /content/qwen3-0.6b: 서버에 올릴 모델 파일이 저장된 경로를 지정합니다.
# 3. --port 8000: 서버가 통신할 포트 번호를 8000번으로 설정합니다.
# 4. --dtype float16: 모델 연산 시 부동소수점(float16) 형식을 사용하여 메모리 사용량을 줄이고 속도를 높입니다.

!python3 -m vllm.entrypoints.openai.api_server \
    --model /content/qwen3-0.6b \
    --port 8000 \
    --dtype float16

2026-02-02 04:39:27.893312: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770007167.948623    7125 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770007167.961761    7125 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770007168.031845    7125 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770007168.033500    7125 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770007168.033526    7125 computation_placer.cc:177] computation placer alr